In [ ]:
import numpy as np
import numpy.linalg as LA
import scipy.optimize as opt
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from tqdm import tqdm
from typing import Callable, Dict, List, Tuple, Optional
import casadi as ca

In [ ]:
# ============================================================
# Data generation
# ============================================================

def sigmoid(z: np.ndarray) -> np.ndarray:
    z = np.clip(z, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-z))


def project_box(x: np.ndarray, l_val: np.ndarray, u_val: np.ndarray) -> np.ndarray:
    return np.clip(x, l_val, u_val)


def generate_fairness_logistic_problem(
    n_samples: int = 5000,
    n_features: int = 200,
    n_groups: int = 8,
    n_group_constraints: int = 100,
    box_radius: float = 1.0,
    margin: float = 0.10,
    seed: int = 0,
) -> Dict[str, np.ndarray]:
    """
    Synthetic constrained logistic regression:

        min_w  (1/N) sum_j log(1 + exp(-y_j x_j^T w))
        s.t.   l <= w <= u
               |c_g^T w| <= tau_g, g = 1,...,m

    The fairness constraints are built from differences of group mean feature vectors.
    """
    rng = np.random.default_rng(seed)

    # group membership
    group_ids = np.tile(np.arange(n_groups), int(np.ceil(n_samples / n_groups)))[:n_samples]
    rng.shuffle(group_ids)

    # make the classification task nontrivial
    group_shifts = rng.normal(scale=0.50, size=(n_groups, n_features))
    X = rng.normal(size=(n_samples, n_features)) + group_shifts[group_ids]

    # box constraints
    l_val = -box_radius * np.ones(n_features)
    u_val = box_radius * np.ones(n_features)

    # ground-truth classifier
    w_true = rng.normal(scale=0.50, size=n_features)
    w_true = project_box(w_true, l_val, u_val)

    # labels in {-1, +1}
    probs = sigmoid(X @ w_true)
    y01 = rng.binomial(1, probs)
    y = 2.0 * y01 - 1.0

    # group means
    group_means = np.zeros((n_groups, n_features))
    for g in range(n_groups):
        idx = np.where(group_ids == g)[0]
        group_means[g] = X[idx].mean(axis=0)

    # fairness constraints |(mean_g1 - mean_g2)^T w| <= tau
    C = np.zeros((n_group_constraints, n_features))
    tau = np.zeros(n_group_constraints)
    for j in range(n_group_constraints):
        g1 = int(rng.integers(0, n_groups))
        g2 = int(rng.integers(0, n_groups))
        while g2 == g1:
            g2 = int(rng.integers(0, n_groups))
        c = group_means[g1] - group_means[g2]
        C[j] = c
        tau[j] = abs(c @ w_true) + margin

    # two-sided linear constraints
    A_constr = np.vstack([C, -C])
    b_constr = np.concatenate([tau, tau])

    return {
        "X": X,
        "y": y,
        "w_true": w_true,
        "A_constr": A_constr,
        "b_constr": b_constr,
        "l_val": l_val,
        "u_val": u_val,
        "group_ids": group_ids,
        "group_means": group_means,
    }


In [ ]:
# ============================================================
# Objective, gradient, constraints, and metrics
# ============================================================

def logistic_loss_and_grad(w: np.ndarray, X: np.ndarray, y: np.ndarray) -> Tuple[float, np.ndarray]:
    """Mean logistic loss and gradient."""
    yz = y * (X @ w)
    loss = float(np.mean(np.logaddexp(0.0, -yz)))
    coeff = -y / (1.0 + np.exp(np.clip(yz, -50.0, 50.0)))
    grad = (X.T @ coeff) / X.shape[0]
    return loss, grad


def compute_function_values(x_hist: List[np.ndarray], X: np.ndarray, y: np.ndarray) -> List[float]:
    return [logistic_loss_and_grad(x, X, y)[0] for x in x_hist]


def compute_infeas_gap_list(
    x_hat_list: List[np.ndarray],
    A_constr: np.ndarray,
    b_constr: np.ndarray,
    l_val: np.ndarray,
    u_val: np.ndarray,
) -> List[float]:
    out = []
    for x in x_hat_list:
        lin_violation = np.maximum(A_constr @ x - b_constr, 0.0)
        box_violation = np.maximum(np.maximum(l_val - x, 0.0), np.maximum(x - u_val, 0.0))
        out.append(float(max(np.max(lin_violation) if lin_violation.size else 0.0, np.max(box_violation))))
    return out


def g_val(i: int, x: np.ndarray, A_constr: np.ndarray, b_constr: np.ndarray) -> float:
    return float(A_constr[i] @ x - b_constr[i])


def g_plus(i: int, x: np.ndarray, A_constr: np.ndarray, b_constr: np.ndarray) -> float:
    return max(0.0, g_val(i, x, A_constr, b_constr))


def subgradient(i: int, x: np.ndarray, A_constr: np.ndarray, b_constr: np.ndarray) -> np.ndarray:
    if g_val(i, x, A_constr, b_constr) > 0.0:
        return A_constr[i].copy()
    return np.zeros_like(x)


def project_Y(x: np.ndarray, l_val: np.ndarray, u_val: np.ndarray) -> np.ndarray:
    return project_box(x, l_val, u_val)


def weighted_running_average(hist_list: List[np.ndarray], gamma: np.ndarray) -> List[np.ndarray]:
    out = []
    num = None
    den = 0.0
    for w, g in zip(hist_list, gamma):
        if num is None:
            num = np.zeros_like(w, dtype=float)
        num = num + g * w
        den += g
        out.append(num / den)
    return out


def compute_mean_std_across_experiments(
    func_runs: List[List[float]],
    infeas_runs: List[List[float]],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    func_arr = np.array(func_runs, dtype=float)
    infeas_arr = np.array(infeas_runs, dtype=float)
    return func_arr.mean(axis=0), func_arr.std(axis=0), infeas_arr.mean(axis=0), infeas_arr.std(axis=0)


In [ ]:

# ============================================================
# Algorithm 1: Randomized feasibility
# ============================================================

def randomized_feasibility(
    v_k: np.ndarray,
    N_k: int,
    beta: float,
    m: int,
    A_constr: np.ndarray,
    b_constr: np.ndarray,
    l_val: np.ndarray,
    u_val: np.ndarray,
) -> np.ndarray:
    z = v_k.copy()
    for _ in range(N_k):
        i = np.random.randint(0, m)
        g_i = g_plus(i, z, A_constr, b_constr)
        d = subgradient(i, z, A_constr, b_constr)
        norm_d2 = float(np.dot(d, d))
        if norm_d2 > 0.0:
            z = z - beta * (g_i / norm_d2) * d
        z = project_Y(z, l_val, u_val)
    return z


# ============================================================
# Algorithm 3: DoWS with randomized feasibility
# ============================================================

def dows_with_randomized_feasibility(
    x0: np.ndarray,
    grad_f: Callable[[np.ndarray], np.ndarray],
    N_schedule: Callable[[int], int],
    A_constr: np.ndarray,
    b_constr: np.ndarray,
    l_val: np.ndarray,
    u_val: np.ndarray,
    T: int,
    r0: float,
    beta_feas: float = 1.0,
) -> Tuple[List[np.ndarray], List[float]]:
    x_k = x0.copy()
    x_0 = x0.copy()
    r_prev = float(r0)
    p_prev = 0.0

    func_bar_x_k: List[float] = []
    bar_x_k_list: List[np.ndarray] = []

    num = np.zeros_like(x0, dtype=float)
    den = 0.0
    p_1 = None

    for k in tqdm(range(T)):
        s_f_k = grad_f(x_k)
        r_k = max(LA.norm(x_k - x_0), r_prev)
        p_k = p_prev + (r_k ** 2) * (LA.norm(s_f_k) ** 2)
        p_k = max(p_k, 1e-16)

        alpha_k = (r_k ** 2) / np.sqrt(p_k)
        v_k = project_Y(x_k - alpha_k * s_f_k, l_val, u_val)

        N_k = int(N_schedule(k))
        x_k = randomized_feasibility(
            v_k,
            N_k,
            beta=beta_feas,
            m=A_constr.shape[0],
            A_constr=A_constr,
            b_constr=b_constr,
            l_val=l_val,
            u_val=u_val,
        )

        num += (r_k ** 2) * x_k
        den += r_k ** 2
        bar_x_k = num / den
        bar_x_k_list.append(bar_x_k.copy())

        func_bar_x_k.append(grad_f_loss(bar_x_k, grad_f))

        r_prev = r_k
        p_prev = p_k
        if p_1 is None:
            p_1 = p_k

    return bar_x_k_list, func_bar_x_k


# ============================================================
# Algorithm 4: T-DoWS with randomized feasibility
# ============================================================

def tdows_with_randomized_feasibility(
    x0: np.ndarray,
    grad_f: Callable[[np.ndarray], np.ndarray],
    N_schedule: Callable[[int], int],
    A_constr: np.ndarray,
    b_constr: np.ndarray,
    l_val: np.ndarray,
    u_val: np.ndarray,
    T: int,
    r0: float,
    beta_feas: float = 1.0,
) -> Tuple[List[np.ndarray], List[float]]:
    x_k = x0.copy()
    x_0 = x0.copy()
    r_prev = float(r0)
    p_prev = 0.0

    func_bar_x_k: List[float] = []
    bar_x_k_list: List[np.ndarray] = []

    num = np.zeros_like(x0, dtype=float)
    den = 0.0
    p_1 = None

    for k in tqdm(range(T)):
        s_f_k = grad_f(x_k)
        r_k = max(LA.norm(x_k - x_0), r_prev)
        p_k = p_prev + (r_k ** 2) * (LA.norm(s_f_k) ** 2)
        p_k = max(p_k, 1e-16)

        if p_1 is None:
            p_1 = p_k
        denom = np.sqrt(2.0 * p_k) * np.log(np.e * p_k / p_1)
        denom = denom if abs(denom) > 1e-16 else 1e-16
        alpha_k = (r_k ** 2) / denom

        v_k = project_Y(x_k - alpha_k * s_f_k, l_val, u_val)

        N_k = int(N_schedule(k))
        x_k = randomized_feasibility(
            v_k,
            N_k,
            beta=beta_feas,
            m=A_constr.shape[0],
            A_constr=A_constr,
            b_constr=b_constr,
            l_val=l_val,
            u_val=u_val,
        )

        num += (r_k ** 2) * x_k
        den += r_k ** 2
        bar_x_k = num / den
        bar_x_k_list.append(bar_x_k.copy())

        func_bar_x_k.append(grad_f_loss(bar_x_k, grad_f))

        r_prev = r_k
        p_prev = p_k

    return bar_x_k_list, func_bar_x_k


# helper for the running objective value using the gradient function closure
# grad_f_loss is attached at runtime by set_loss_function.
_grad_loss_fn = None

def set_loss_function(loss_fn: Callable[[np.ndarray], float]) -> None:
    global _grad_loss_fn
    _grad_loss_fn = loss_fn


def grad_f_loss(x: np.ndarray, grad_f: Callable[[np.ndarray], np.ndarray]) -> float:
    if _grad_loss_fn is not None:
        return float(_grad_loss_fn(x))
    raise RuntimeError("Call set_loss_function(loss_fn) before running DoWS/T-DoWS.")

In [ ]:
# ============================================================
# Arrow-Hurwicz
# ============================================================

def arrow_hurwicz(
    x0: np.ndarray,
    lam0: np.ndarray,
    grad_f: Callable[[np.ndarray], np.ndarray],
    alpha_schedule: Callable[[int], float],
    tau_schedule: Callable[[int], float],
    A_constr: np.ndarray,
    b_constr: np.ndarray,
    l_val: np.ndarray,
    u_val: np.ndarray,
    T: int,
):
    x_k = x0.copy()
    lam_k = lam0.copy()

    x_list = []
    lam_list = []

    for k in tqdm(range(T)):
        alpha_k = float(alpha_schedule(k))
        eta_k = float(tau_schedule(k))

        grad_x = grad_f(x_k)
        grad_x = grad_x + A_constr.T @ lam_k
        x_next = project_Y(x_k - alpha_k * grad_x, l_val, u_val)

        lam_next = np.maximum(0.0, lam_k + eta_k * (A_constr @ x_next - b_constr))

        x_k = x_next
        lam_k = lam_next
        x_list.append(x_k.copy())
        lam_list.append(lam_k.copy())

    return x_list, lam_list


# def alt_gda(
#     x0: np.ndarray,
#     lam0: np.ndarray,
#     grad_f: Callable[[np.ndarray], np.ndarray],
#     alpha_schedule: Callable[[int], float],
#     tau_schedule: Callable[[int], float],
#     A_constr: np.ndarray,
#     b_constr: np.ndarray,
#     l_val: np.ndarray,
#     u_val: np.ndarray,
#     T: int,
# ):
#     """Alternating GDA wrapper with the same primal-dual structure."""
#     x_k = x0.copy()
#     lam_k = lam0.copy()

#     x_list = []
#     lam_list = []

#     for k in tqdm(range(T)):
#         alpha_k = float(alpha_schedule(k))
#         eta_k = float(tau_schedule(k))

#         # primal step
#         grad_x = grad_f(x_k) + A_constr.T @ lam_k
#         x_next = project_Y(x_k - alpha_k * grad_x, l_val, u_val)

#         # dual step (uses the updated primal variable)
#         lam_next = np.maximum(0.0, lam_k + eta_k * (A_constr @ x_next - b_constr))

#         x_k = x_next
#         lam_k = lam_next
#         x_list.append(x_k.copy())
#         lam_list.append(lam_k.copy())

#     return x_list, lam_list


# ========================================================================
# ACVI for the fairness problem (code has some issue -- need to be fixed)
# ========================================================================

# def acvi_fairness(
#     y0: np.ndarray,
#     lam0: np.ndarray,
#     X: np.ndarray,
#     y_labels: np.ndarray,
#     A_constr: np.ndarray,
#     b_constr: np.ndarray,
#     l_val: np.ndarray,
#     u_val: np.ndarray,
#     mu_init: float,
#     beta: float,
#     delta: float,
#     T: int,
#     K: int,
#     verbose: bool = False,
#     barrier_tol: float = 1e-16,
# ):
#     """
#     ACVI-style baseline adapted to logistic loss.

#     This is not the exact QCQP inner linear-system update from the notebook;
#     instead, it uses a barrier-regularized x-update and a dual update.
#     """
#     y_k_t = y0.copy().astype(float)
#     lam_k_t = lam0.copy().astype(float)
#     n = y_k_t.size

#     y_hist, lam_hist = [], []
#     x_last_per_t, x_mean_per_t = [], []

#     mu_prev = float(mu_init)

#     def obj_barrier(x: np.ndarray, c: np.ndarray, mu_t: float) -> float:
#         loss, _ = logistic_loss_and_grad(x, X, y_labels)
#         g_main = A_constr @ x - b_constr
#         if np.any(g_main >= -barrier_tol):
#             return 1e12 + 1e6 * float(np.maximum(g_main.max(), 0.0))
#         barrier = -mu_t * np.sum(np.log(-g_main))
#         prox = 0.5 * beta * np.sum((x - c) ** 2)
#         return loss + barrier + prox

#     def grad_obj_barrier(x: np.ndarray, c: np.ndarray, mu_t: float) -> np.ndarray:
#         loss, grad = logistic_loss_and_grad(x, X, y_labels)
#         g_main = A_constr @ x - b_constr
#         inv = 1.0 / np.maximum(-g_main, 1e-12)
#         grad_bar = mu_t * (A_constr.T @ inv)
#         grad_prox = beta * (x - c)
#         return grad + grad_bar + grad_prox

#     for t in tqdm(range(T), desc="Outer loop"):
#         mu_t = delta * mu_prev
#         mu_prev = mu_t
#         if verbose:
#             print(f"[Outer {t}] mu_t={mu_t:.3e}")

#         x_sum = np.zeros(n)

#         for _ in range(K):
#             # x-update with box bounds
#             center = y_k_t - (1.0 / beta) * lam_k_t
#             res = opt.minimize(
#                 fun=lambda z: obj_barrier(z, center, mu_t),
#                 x0=y_k_t,
#                 jac=lambda z: grad_obj_barrier(z, center, mu_t),
#                 method="L-BFGS-B",
#                 bounds=list(zip(l_val, u_val)),
#                 options={"maxiter": 300, "ftol": 1e-10},
#             )
#             x_next = res.x
#             x_sum += x_next

#             # dual update
#             lam_next = np.maximum(0.0, lam_k_t + beta * (A_constr @ x_next - b_constr))

#             y_k_t = x_next.copy()
#             lam_k_t = lam_next.copy()

#         x_last_per_t.append(x_next.copy())
#         x_mean_per_t.append(x_sum / float(K))
#         y_hist.append(y_k_t.copy())
#         lam_hist.append(lam_k_t.copy())

#     return x_last_per_t, x_mean_per_t, y_hist, lam_hist

In [ ]:
# ============================================================
# Ipopt code via Casadi
# ============================================================

def solve_with_ipopt_casadi(
    X: np.ndarray,
    y_labels: np.ndarray,
    A_constr: np.ndarray,
    b_constr: np.ndarray,
    l_val: np.ndarray,
    u_val: np.ndarray,
    x0: Optional[np.ndarray] = None,
    tee: bool = True,
) -> Tuple[np.ndarray, float, dict]:
    """
    Solve the fairness logistic regression problem with CasADi + Ipopt:

        min_x  (1/N) * sum_j log(1 + exp(-y_j * <X_j, x>))
        s.t.   A_constr x - b_constr <= 0
               l_val <= x <= u_val

    Returns:
        x_opt  : optimal solution as a NumPy array
        obj_val: optimal objective value
        sol    : raw CasADi solution object
    """
    X_dm = ca.DM(X)
    y_dm = ca.DM(y_labels).reshape((X.shape[0], 1))
    A_dm = ca.DM(A_constr)
    b_dm = ca.DM(b_constr).reshape((A_constr.shape[0], 1))

    lbx = np.asarray(l_val, dtype=float).flatten()
    ubx = np.asarray(u_val, dtype=float).flatten()

    N, n = X.shape
    m = A_constr.shape[0]

    x = ca.MX.sym("x", n, 1)

    # Logistic loss
    logits = ca.mtimes(X_dm, x)                    # shape: (N, 1)
    yz = y_dm * logits                 # elementwise
    obj = ca.sum1(ca.log(1 + ca.exp(-yz))) / N

    # Linear inequality constraints: A_constr x - b_constr <= 0
    g = ca.mtimes(A_dm, x) - b_dm

    nlp = {"x": x, "f": obj, "g": g}

    opts = {
        "ipopt.print_level": 5 if tee else 0,
        "print_time": bool(tee),
        "ipopt.tol": 1e-8,
        "ipopt.max_iter": 2000,
    }

    solver = ca.nlpsol("solver", "ipopt", nlp, opts)

    if x0 is None:
        x0 = np.zeros(n, dtype=float)
    else:
        x0 = np.asarray(x0, dtype=float).flatten()

    sol = solver(
        x0=x0,
        lbx=lbx,
        ubx=ubx,
        lbg=-np.inf * np.ones(m),
        ubg=np.zeros(m),
    )

    x_opt = np.array(sol["x"]).flatten()
    obj_val = float(sol["f"])

    return x_opt, obj_val, sol

In [ ]:
data = generate_fairness_logistic_problem(
    n_samples=10000,
    n_features=500,
    n_groups=10,
    n_group_constraints= 1000, #10000,
    box_radius=1.0,
    margin=0.10,
    seed=42,
)

X = data["X"]
y_labels = data["y"]
A_constr = data["A_constr"]
b_constr = data["b_constr"]
l_val = data["l_val"]
u_val = data["u_val"]

# f_true, _ = logistic_loss_and_grad(data["w_true"], X, y_labels)
# print("f_true: ", f_true)

# objective helpers
def loss_fn(x: np.ndarray) -> float:
    return logistic_loss_and_grad(x, X, y_labels)[0]

def grad_f(x: np.ndarray) -> np.ndarray:
    return logistic_loss_and_grad(x, X, y_labels)[1]

set_loss_function(loss_fn)

# feasible starting point
x0 = project_Y(data["w_true"].copy(), l_val, u_val)
lam0 = np.zeros(A_constr.shape[0])

# -----------------------------------------------------------------
# Schedules (edit these from your experiment cell)
# -----------------------------------------------------------------
Tot_itr = 1000

def N_schedule_log(k: int) -> int:
    return int(np.ceil(np.log(k + 2)))

def N_schedule_sqrt(k: int) -> int:
    return int(np.ceil(np.sqrt(k + 1)))

def N_schedule_new(k: int) -> int:
    return 100 + int(np.ceil((k + 1)**(1/1.1)))

def alpha_schedule_ah(k: int) -> float:
    return 1.0 / np.sqrt(Tot_itr)

def tau_schedule_ah(k: int) -> float:
    return 1.0 / np.sqrt(Tot_itr)

In [ ]:
# pip install casadi

In [ ]:
# -----------------------------------------------------------------
# Ipopt (Casadi)
# -----------------------------------------------------------------
x_opt, obj_val, sol = solve_with_ipopt_casadi(
    X=X,
    y_labels=y_labels,
    A_constr=A_constr,
    b_constr=b_constr,
    l_val=l_val,
    u_val=u_val,
    x0=x0,
    tee=True,
)

print("Optimal value:", obj_val)
#print("Optimal x:", x_opt)

# infeas_ipopt = compute_infeas_gap_list(x_opt, A_constr, b_constr, l_val, u_val)

In [ ]:
lin_violation_ipopt = np.maximum(A_constr @ x_opt - b_constr, 0.0)
box_violation_ipopt = np.maximum(np.maximum(l_val - x_opt, 0.0), np.maximum(x_opt - u_val, 0.0))
infeas_ipopt = float(max(np.max(lin_violation_ipopt) if lin_violation_ipopt.size else 0.0, np.max(box_violation_ipopt)))
print("Infeasibility IPOPT:", infeas_ipopt)

In [ ]:
no_of_experiments = 5

# -----------------------------------------------------------------
# DoWS / T-DoWS
# -----------------------------------------------------------------

func_dows_sqrt = []
infeas_dows_sqrt = []

for i in range(no_of_experiments):
  x_dows, f_dows = dows_with_randomized_feasibility(
      x0=x0,
      grad_f=grad_f,
      N_schedule=N_schedule_new,
      A_constr=A_constr,
      b_constr=b_constr,
      l_val=l_val,
      u_val=u_val,
      T=Tot_itr,
      r0=1e-1,
      beta_feas=1.0,
  )
  inf_dows = compute_infeas_gap_list(x_dows, A_constr, b_constr, l_val, u_val)
  func_dows_sqrt.append(np.asarray(f_dows, dtype=float))
  infeas_dows_sqrt.append(np.asarray(inf_dows, dtype=float))


func_tdows_sqrt = []
infeas_tdows_sqrt = []
for i in range(no_of_experiments):
  x_tdows, f_tdows = tdows_with_randomized_feasibility(
      x0=x0,
      grad_f=grad_f,
      N_schedule=N_schedule_new,
      A_constr=A_constr,
      b_constr=b_constr,
      l_val=l_val,
      u_val=u_val,
      T=Tot_itr,
      r0=1e-1,
      beta_feas=1.0,
  )
  inf_tdows = compute_infeas_gap_list(x_tdows, A_constr, b_constr, l_val, u_val)
  func_tdows_sqrt.append(np.asarray(f_tdows, dtype=float))
  infeas_tdows_sqrt.append(np.asarray(inf_tdows, dtype=float))

In [ ]:


# -----------------------------------------------------------------
# Arrow-Hurwicz
# -----------------------------------------------------------------
x_ah, lam_ah = arrow_hurwicz(
    x0=x0,
    lam0=lam0,
    grad_f=grad_f,
    alpha_schedule=alpha_schedule_ah,
    tau_schedule=tau_schedule_ah,
    A_constr=A_constr,
    b_constr=b_constr,
    l_val=l_val,
    u_val=u_val,
    T=Tot_itr,
)
x_ah_bar = weighted_running_average(x_ah, np.ones(len(x_ah)))
f_ah = compute_function_values(x_ah_bar, X, y_labels)
inf_ah = compute_infeas_gap_list(x_ah_bar, A_constr, b_constr, l_val, u_val)


In [ ]:
# # ============================================================
# # For Plotting
# # ============================================================


def compute_mean_std_across_experiments(func_logs, infeas_logs):
    """
    Stack results across experiments and compute mean and std along axis=0.

    Args:
        func_logs  : list of lists or arrays
                     func_logs[i] = list/array of function values from experiment i
        infeas_logs: list of lists or arrays
                     infeas_logs[i] = list/array of infeasibility values from experiment i

    Returns:
        mean_func, std_func, mean_infeas, std_infeas : np.ndarray
            mean_func   : mean of function values across experiments
            std_func    : standard deviation of function values across experiments
            mean_infeas : mean of infeasibility values across experiments
            std_infeas  : standard deviation of infeasibility values across experiments
    """
    # Find the minimum common length (in case experiments differ slightly)
    min_len = min(len(arr) for arr in func_logs)

    # Stack up to the min length
    func_stack = np.vstack([np.array(arr[:min_len]) for arr in func_logs])
    infeas_stack = np.vstack([np.array(arr[:min_len]) for arr in infeas_logs])

    # Compute statistics across experiments
    mean_func = np.mean(func_stack, axis=0)
    std_func  = np.std(func_stack, axis=0)
    mean_infeas = np.mean(infeas_stack, axis=0)
    std_infeas  = np.std(infeas_stack, axis=0)

    return mean_func, std_func, mean_infeas, std_infeas



mean_func_dows_sqrt, std_func_dows_sqrt, mean_infeas_dows_sqrt, std_infeas_dows_sqrt = compute_mean_std_across_experiments(
    func_dows_sqrt, infeas_dows_sqrt
    )

mean_func_tdows_sqrt, std_func_tdows_sqrt, mean_infeas_tdows_sqrt, std_infeas_tdows_sqrt = compute_mean_std_across_experiments(
    func_tdows_sqrt, infeas_tdows_sqrt
    )

In [ ]:
iters = np.arange(1, Tot_itr + 1)

figure(1)

plt.plot(iters, mean_func_dows_sqrt, color='#009E73', label='Algorithm 3')
plt.fill_between(iters, mean_func_dows_sqrt - std_func_dows_sqrt,
                 mean_func_dows_sqrt + std_func_dows_sqrt, color='#009E73', alpha=0.2)

plt.plot(iters, mean_func_tdows_sqrt, color='#CC79A7', label='Algorithm 4')
plt.fill_between(iters, mean_func_tdows_sqrt - std_func_tdows_sqrt,
                 mean_func_tdows_sqrt + std_func_tdows_sqrt, color='#CC79A7', alpha=0.2)


plt.plot(iters, f_ah, color='#56B4E9', linestyle='-', label='Arrow–Hurwicz', marker = '^', markevery = 100, alpha = 0.6)

# For IPOPT
func_ipopt = obj_val*np.ones(Tot_itr)
plt.plot(iters, func_ipopt, color='magenta', linestyle='--', label='IPOPT', marker = '^', markevery = 100, alpha = 0.6)

plt.xlabel('Number of iterations', fontsize=18)
plt.ylabel(r'$f(\bar x_k)$', fontsize=18)
#plt.title(r'$|f(\bar x_k) - f^*|$ ($f$ convex, $f^*$ unknown)', fontsize=18)
plt.yscale('log')
#plt.ylim(-0.4, -0.25)
plt.grid(True)
plt.legend(fontsize=12, loc='best')
plt.tight_layout()
plt.show()

In [ ]:
figure(2)

plt.plot(iters, mean_infeas_dows_sqrt, color='#009E73', label='Algorithm 3')
plt.fill_between(iters, mean_infeas_dows_sqrt - std_infeas_dows_sqrt,
                 mean_infeas_dows_sqrt + std_infeas_dows_sqrt, color='#009E73', alpha=0.2)

plt.plot(iters, mean_infeas_tdows_sqrt, color='#CC79A7', label='Algorithm 4')
plt.fill_between(iters, mean_infeas_tdows_sqrt - std_infeas_tdows_sqrt,
                 mean_infeas_tdows_sqrt + std_infeas_tdows_sqrt, color='#CC79A7', alpha=0.2)

plt.plot(iters, inf_ah, color='#56B4E9', linestyle='-', label='Arrow–Hurwicz', marker = '^', markevery = 5, alpha = 0.5)

infs_ipopt = infeas_ipopt * np.ones(Tot_itr)
plt.plot(iters, infs_ipopt, color='magenta', linestyle='--', label='IPOPT', marker = '^', markevery = 100, alpha = 0.6)

plt.xlabel('Number of iterations', fontsize=18)
plt.ylabel(r'Infeasibility (max violation)', fontsize=18)
plt.yscale('log')
plt.grid(True)
plt.legend(fontsize=12, loc='best')

plt.tight_layout()
plt.show()